# Advanced PEFT Adapters in Ludwig

This notebook demonstrates Ludwig's extended parameter-efficient fine-tuning (PEFT) support:

- **Advanced LoRA initializers**: PiSSA, EVA, CorDA, LoftQ, OLoRA, orthogonal
- **LoRA extras**: rsLoRA, DoRA, LoRA+, per-layer rank/alpha patterns, layer replication
- **TinyLoRA** — LoRA-XS variant: SVD + fixed projections, as few as 13 parameters
- **C3A** — contextual/conditional/compositional adapter
- **OFT** — orthogonal fine-tuning (preserves hyperspherical energy)
- **HRA** — Householder Reflection Adaptation
- **WaveFT** — wavelet-domain fine-tuning
- **LN-Tuning** — fine-tune only layer normalization parameters
- **VBLoRA** — vector bank LoRA (shared vector reuse across layers)

**References**: Issue [#4129](https://github.com/ludwig-ai/ludwig/issues/4129)

In [ ]:
# Install dependencies
# !pip install ludwig peft>=0.19.0 transformers torch

## 1. Advanced LoRA Initializers

The `init_lora_weights` field selects how the LoRA A and B matrices are initialized.
The default (`True`) uses Kaiming uniform (A) and zeros (B).
Advanced initializers often converge faster or reach higher final performance.

In [ ]:
from ludwig.schema.llms.peft import adapter_registry, LoraConfig

# Standard LoRA (baseline)
standard = LoraConfig(r=8, alpha=16)
print("Standard LoRA init_lora_weights:", standard.to_config("CAUSAL_LM").init_lora_weights)

# PiSSA: Principal Singular values and Singular vectors Adaptation
# Initializes A, B from SVD of the pretrained weight. Faster convergence.
pissa = LoraConfig(r=8, alpha=16, init_lora_weights="pissa")
print("PiSSA init_lora_weights:", pissa.to_config("CAUSAL_LM").init_lora_weights)

# CorDA: Context-Oriented Decomposition Adaptation
corda = LoraConfig(r=8, alpha=16, init_lora_weights="corda")
print("CorDA init_lora_weights:", corda.to_config("CAUSAL_LM").init_lora_weights)

In [ ]:
from ludwig.schema.llms.peft import LoraConfig

# EVA: Explained Variance Adaptation — data-driven initialization from activation SVD
eva_cfg = LoraConfig.model_validate(
    {
        "type": "lora",
        "r": 16,
        "init_lora_weights": "eva",
        "eva_config": {"rho": 2.0, "tau": 0.99, "adjust_scaling_factors": True},
    }
)
peft_cfg = eva_cfg.to_config("CAUSAL_LM")
print("EVA init:", peft_cfg.init_lora_weights)
print("EVA config rho:", peft_cfg.eva_config.rho)

In [ ]:
from ludwig.schema.llms.peft import LoraConfig

# LoftQ: jointly quantizes base weights and initializes LoRA to minimize error
loftq_cfg = LoraConfig.model_validate(
    {"type": "lora", "r": 16, "init_lora_weights": "loftq", "loftq_config": {"loftq_bits": 4, "loftq_iter": 1}}
)
peft_cfg = loftq_cfg.to_config("CAUSAL_LM")
print("LoftQ init:", peft_cfg.init_lora_weights)
print("LoftQ bits:", peft_cfg.loftq_config["loftq_bits"])

## 2. Per-Layer Rank Patterns

Different layers may benefit from different ranks. Use `rank_pattern` and `alpha_pattern`
to override the global `r` and `alpha` for specific layers.

In [ ]:
from ludwig.schema.llms.peft import LoraConfig

cfg = LoraConfig.model_validate(
    {
        "type": "lora",
        "r": 8,  # global default rank
        "alpha": 16,
        "init_lora_weights": "pissa",
        # Override specific layers: attention layers get higher rank
        "rank_pattern": {
            "model.layers.0.self_attn.q_proj": 16,
            "model.layers.0.self_attn.v_proj": 16,
        },
        "alpha_pattern": {
            "model.layers.0.self_attn.q_proj": 32.0,
        },
    }
)
peft_cfg = cfg.to_config("CAUSAL_LM")
print("rank_pattern:", peft_cfg.rank_pattern)
print("alpha_pattern:", peft_cfg.alpha_pattern)

## 3. TinyLoRA — Extreme Parameter Efficiency

TinyLoRA ([arXiv:2602.04118](https://arxiv.org/abs/2602.04118)) achieves fine-tuning with
as few as 13 parameters. It uses SVD decomposition of frozen weights and projects a tiny
trainable vector through fixed random tensors.

In [ ]:
from ludwig.schema.llms.peft import TinyLoraAdapterConfig

# Standard TinyLoRA: r=2, u=64 (like LoRA-XS)
tinylora = TinyLoraAdapterConfig.model_validate({"type": "tinylora", "r": 2, "u": 64})
print("TinyLoRA config:", tinylora)

# Ultra-minimal: r=2, u=13 (13 parameters per layer group!)
micro = TinyLoraAdapterConfig.model_validate({"type": "tinylora", "r": 2, "u": 13})
peft_cfg = micro.to_config("CAUSAL_LM")
print("\nUltra-minimal u:", peft_cfg.u)

In [ ]:
# Ludwig YAML config equivalent:
tinylora_yaml = """
model_type: llm
base_model: meta-llama/Llama-3.2-1B

input_features:
  - name: instruction
    type: text

output_features:
  - name: output
    type: text

adapter:
  type: tinylora
  r: 2
  u: 64
  weight_tying: 0.0  # 0.5 shares vectors across 50% of modules

trainer:
  type: finetune
  learning_rate: 0.001
  epochs: 3
"""
print(tinylora_yaml)

## 4. C3A — Contextual/Conditional/Compositional Adapter

In [ ]:
from ludwig.schema.llms.peft import C3AAdapterConfig

c3a = C3AAdapterConfig.model_validate({"type": "c3a", "block_size": 256})
peft_cfg = c3a.to_config("CAUSAL_LM")
print("C3A block_size:", peft_cfg.block_size)

## 5. Orthogonal Methods: OFT and HRA

OFT and HRA apply orthogonal transformations to weight matrices, preserving the
hyperspherical energy of the pre-trained model. This prevents catastrophic forgetting
and is particularly effective for subject-driven generation.

In [ ]:
from ludwig.schema.llms.peft import HRAAdapterConfig, OFTAdapterConfig

# OFT: Orthogonal Fine-Tuning
oft = OFTAdapterConfig.model_validate({"type": "oft", "oft_block_size": 32})
print("OFT:", oft.to_config("CAUSAL_LM"))

# COFT variant (constrained OFT):
coft = OFTAdapterConfig.model_validate({"type": "oft", "coft": True, "eps": 6e-5})
peft_cfg = coft.to_config("CAUSAL_LM")
print("\nCOFT enabled:", peft_cfg.coft)

# HRA: Householder Reflection Adaptation
hra = HRAAdapterConfig.model_validate({"type": "hra", "r": 8, "apply_GS": True})
print("\nHRA r:", hra.r, "apply_GS:", hra.apply_GS)

## 6. WaveFT — Wavelet-Domain Fine-Tuning

In [ ]:
from ludwig.schema.llms.peft import WaveFTAdapterConfig

# db1 (Haar) wavelet — simplest, fastest
waveft = WaveFTAdapterConfig.model_validate({"type": "waveft", "wavelet_family": "db1", "n_frequency": 2592})
print("WaveFT wavelet:", waveft.wavelet_family, "n_frequency:", waveft.n_frequency)

## 7. LN-Tuning — Ultra-Lightweight Adaptation

In [ ]:
from ludwig.schema.llms.peft import LNTuningAdapterConfig

# Fine-tune only LayerNorm/RMSNorm params — typically <0.1% of total params
ln = LNTuningAdapterConfig.model_validate({"type": "ln_tuning"})
print("LN-Tuning target_modules:", ln.target_modules, "(None = all LayerNorms)")

## 8. VBLoRA — Vector Bank LoRA

In [ ]:
from ludwig.schema.llms.peft import VBLoRAAdapterConfig

# VBLoRA: shared vector bank across all layers
vblora = VBLoRAAdapterConfig.model_validate(
    {
        "type": "vblora",
        "r": 4,
        "num_vectors": 256,
        "vector_length": 4096,  # set to model hidden size
        "topk": 2,
    }
)
peft_cfg = vblora.to_config("CAUSAL_LM")
print("VBLoRA r:", peft_cfg.r, "num_vectors:", peft_cfg.num_vectors, "topk:", peft_cfg.topk)

## 9. Full End-to-End Training Example

Here's how to use these adapters in a complete Ludwig training run:

In [ ]:
import yaml

# Choose your adapter:
adapter_configs = {
    "pissa": {"type": "lora", "r": 16, "alpha": 32, "init_lora_weights": "pissa"},
    "tinylora": {"type": "tinylora", "r": 2, "u": 64},
    "ln_tuning": {"type": "ln_tuning"},
    "oft": {"type": "oft", "oft_block_size": 32},
}

selected = "pissa"

config = {
    "model_type": "llm",
    "base_model": "meta-llama/Llama-3.2-1B",
    "input_features": [{"name": "instruction", "type": "text"}],
    "output_features": [{"name": "output", "type": "text"}],
    "adapter": adapter_configs[selected],
    "trainer": {
        "type": "finetune",
        "learning_rate": 1e-4,
        "epochs": 3,
        "batch_size": 4,
    },
    "preprocessing": {"global_max_sequence_length": 512},
}

print(f"Training with {selected} adapter:")
print(yaml.dump(config, default_flow_style=False))

In [ ]:
# Uncomment to run training:
# from ludwig.api import LudwigModel
# model = LudwigModel(config=config)
# train_stats, _, output_dir = model.train(dataset="path/to/dataset.csv")
# print("Results saved to:", output_dir)

## 10. All Available Adapters

Check which adapter types are registered:

In [ ]:
print("Registered PEFT adapters:")
for name, cls in adapter_registry.items():
    print(f"  {name:<20} -> {cls.__name__}")

## Summary

| Adapter | Key benefit | Params vs. LoRA r=8 |
|---------|-------------|---------------------|
| `lora` + PiSSA | Faster convergence, same params | Same |
| `lora` + EVA | SOTA via data-driven init | Same |
| `lora` + CorDA | Fast convergence + knowledge preservation | Same |
| `lora` + LoftQ | Better QLoRA starting point | Same |
| `tinylora` r=2, u=64 | ~10x fewer params than LoRA | ~10x fewer |
| `tinylora` r=2, u=13 | 13 params per layer group | ~100x fewer |
| `c3a` | Multi-task modular composition | Varies |
| `oft` | Preserves pretrained knowledge | Comparable |
| `hra` | Orthogonal, fewer hyperparams | Comparable |
| `waveft` | Frequency-domain inductive bias | Varies |
| `ln_tuning` | Ultra-lightweight domain shift | <0.1% of total |
| `vblora` | Layer-shared vectors, extreme compression | 10-100x fewer |
